In [ ]:
import sys
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path.cwd().parent))

from src.drift_detector import (
    DRIFT_DIR,
    FEATURE_COLS,
    build_churn_datasets,
    build_revenue_datasets,
    run_data_drift_report,
    run_data_quality_report,
    run_revenue_drift_report,
    run_target_drift_report,
    _extract_drift_summary,
)
from evidently.legacy.pipeline.column_mapping import ColumnMapping

In [ ]:
# Build reference and current churn feature datasets
ref_churn, cur_churn = build_churn_datasets()

print(f"Reference (before 2011-06-01): {len(ref_churn)} customers")
print(f"Current   (before 2011-09-01): {len(cur_churn)} customers")
print(f"\nFeature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print("\nReference feature stats:")
ref_churn[FEATURE_COLS].describe().round(2)

In [ ]:
# Build reference and current revenue datasets
ref_rev, cur_rev = build_revenue_datasets()

print(f"Reference rows: {len(ref_rev)}  (up to 2010-12-31)")
print(f"Current rows:   {len(cur_rev)}  (from 2011-01-01)")
print("\nRevenue stats — reference vs current:")
import pandas as pd
pd.concat([ref_rev.describe().round(2).add_suffix('_ref'),
           cur_rev.describe().round(2).add_suffix('_cur')], axis=1)

In [ ]:
# Data Drift — 13 churn features
r_data, p_data = run_data_drift_report(ref_churn, cur_churn, DRIFT_DIR / "data_drift.html")

try:
    r_data.show()
except Exception:
    display(HTML(open(p_data).read()))

In [ ]:
# Target Drift — predicted_churn + churn_probability
# ColumnMapping is required so Evidently uses the correct target/prediction columns
r_target, p_target = run_target_drift_report(ref_churn, cur_churn, DRIFT_DIR / "target_drift.html")

try:
    r_target.show()
except Exception:
    display(HTML(open(p_target).read()))

In [ ]:
# Data Quality — missing values, outliers, distributions
r_quality, p_quality = run_data_quality_report(ref_churn, cur_churn, DRIFT_DIR / "data_quality.html")

try:
    r_quality.show()
except Exception:
    display(HTML(open(p_quality).read()))

In [ ]:
# Revenue Drift — Revenue + rolling means
r_revenue, p_revenue = run_revenue_drift_report(ref_rev, cur_rev, DRIFT_DIR / "revenue_drift.html")

try:
    r_revenue.show()
except Exception:
    display(HTML(open(p_revenue).read()))

In [ ]:
# Summary table — per-feature drift scores
import pandas as pd

data_summary = _extract_drift_summary(r_data)
rev_summary  = _extract_drift_summary(r_revenue)
target_res   = r_target.as_dict()["metrics"][0]["result"]

rows = []
for col, info in data_summary["per_feature"].items():
    rows.append({
        "feature":        col,
        "drift_detected": info["drift_detected"],
        "drift_score":    round(info["drift_score"], 4),
        "stattest":       info["stattest"],
    })

summary_df = pd.DataFrame(rows).sort_values("drift_detected", ascending=False)

print(f"Churn feature drift share : {data_summary['share_of_drifted_columns']:.4f}")
print(f"Target drift detected     : {target_res['drift_detected']}")
print(f"Revenue drift share       : {rev_summary['share_of_drifted_columns']:.4f}")
print()
summary_df